In [1]:
import os
import matplotlib.pyplot as plt
import numpy as np
import random
import time
import math

import scipy
from scipy import stats

In [2]:
traindata = np.loadtxt("Data_OneStepAhead/Sunspot/train7.txt")
testdata = np.loadtxt("Data_OneStepAhead/Sunspot/test7.txt") 

In [3]:
# An example of a class
class Network:
    def __init__(self, Topo, Train, Test, MaxTime, MinPer):

        self.Top = Topo  # NN topology [input, hidden, output]
        self.Max = MaxTime  # max epocs
        self.TrainData = Train
        self.TestData = Test
        self.NumSamples = Train.shape[0]

        self.lrate = 0  # will be updated later with BP call

        self.minPerf = MinPer
        # initialize weights ( W1 W2 ) and bias ( b1 b2 ) of the network
        np.random.seed()
        self.W1 = np.zeros((self.Top[0], self.Top[1])  )
        self.B1 = np.zeros(self.Top[1])    # bias first layer
        self.W2 = np.zeros((self.Top[1], self.Top[2]) )
        self.B2 = np.zeros(self.Top[2])    # bias second layer
        self.hidout = np.zeros((self.Top[1]))  # output of first hidden layer
        self.out = np.zeros((self.Top[2]))  # output last layer

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def sampleEr(self, actualout):
        error = np.subtract(self.out, actualout)
        sqerror = np.sum(np.square(error)) / self.Top[2]
        return sqerror

    def ForwardPass(self, X):
        z1 = X.dot(self.W1) - self.B1
        self.hidout = self.sigmoid(z1)  # output of first hidden layer#
        z2 = self.hidout.dot(self.W2) #- self.B2
        self.out = self.sigmoid(z2)  # output second hidden layer

    def BackwardPass(self, Input, desired):

        out_delta = (desired - self.out) * (self.out * (1 - self.out))
        hid_delta = np.zeros(self.Top[2])
        hid_delta = out_delta.dot(self.W2.T) * (self.hidout * (1 - self.hidout))

    # update weights and bias
        layer = 1  # hidden to output
        for x in range(0, self.Top[layer]):
            for y in range(0, self.Top[layer + 1]):
                self.W2[x, y] += self.lrate * out_delta[y] * self.hidout[x]
        for y in range(0, self.Top[layer + 1]):
            self.B2[y] += -1 * self.lrate * out_delta[y]

        layer = 0  # Input to Hidden

        for x in range(0, self.Top[layer]):
            for y in range(0, self.Top[layer + 1]):
                self.W1[x, y] += self.lrate * hid_delta[y] * Input[x]
        for y in range(0, self.Top[layer + 1]):
            self.B1[y] += -1 * self.lrate * hid_delta[y]


    def decode(self, w):
        w_layer1size = self.Top[0] * self.Top[1]
        w_layer2size =  self.Top[1] *  self.Top[2]

        w_layer1 = w[0:w_layer1size]
        self.W1 = np.reshape(w_layer1, ( self.Top[0],  self.Top[1]))

        w_layer2 = w[w_layer1size:w_layer1size + w_layer2size]
        self.W2 = np.reshape(w_layer2, ( self.Top[1],  self.Top[2]))
        self.B1 = w[w_layer1size + w_layer2size:w_layer1size + w_layer2size +  self.Top[1]]
        self.B2 = w[w_layer1size + w_layer2size + self.Top[1]:w_layer1size + w_layer2size + self.Top[1] + self.Top[2]]
    def encode(self):
        w1 = self.W1.ravel()
        w2 = self.W2.ravel()
        w = np.concatenate([w1, w2, self.B1, self.B2])
        return w
    def net_size(self):
        return ((self.Top[0] * self.Top[1]) + (self.Top[1] * self.Top[2]) + self.Top[1] + self.Top[2])
    def evaluate_proposal(self,   w ):  # BP with SGD (Stocastic BP)

        self.decode(w)  # method to decode w into W1, W2, B1, B2.
        size = self.TrainData.shape[0]
        Input = np.zeros((1, self.Top[0]))  # temp hold input
        fx = np.zeros(size)



        for pat in range(0, size):
            Input[:] =  self.TrainData[pat, 0:self.Top[0]]
            self.ForwardPass(Input)
            fx[pat] = self.out
        return fx

    def test_proposal(self,  w):  # BP with SGD (Stocastic BP)

        self.decode(w)  # method to decode w into W1, W2, B1, B2.

        size = self.TestData.shape[0]
        Input = np.zeros((1, self.Top[0]))  # temp hold input
        Desired = np.zeros((1, self.Top[2]))
        fx = np.zeros(size)

        sse = 0

        for pat in range(0, size):
            Input[:] = self.TestData[pat, 0:self.Top[0]]
            Desired[:] = self.TestData[pat, self.Top[0]:]
            self.ForwardPass(Input)
            fx[pat] = self.out
            sse = sse + self.sampleEr(Desired)

        rmse = np.sqrt(sse / size)


        return [fx,rmse]


In [4]:
input = 7  # max input
hidden = 5
output = 1
baseNet = [input, hidden, output]
#secondnet = [input, hidden+1, output]
mtaskNet = np.array([baseNet, baseNet])
dims = len(mtaskNet)

In [5]:
Net = Network(mtaskNet[0], traindata, testdata, 0.1, 0.0002)
netsize = Net.net_size()
w = np.random.randn(netsize)
print(Net.evaluate_proposal(w))

[0.50989621 0.50998664 0.51037028 0.51305038 0.51594249 0.51934784
 0.52528894 0.53240543 0.54170926 0.55075703 0.55736158 0.56298958
 0.56685255 0.56873198 0.56823484 0.56777015 0.57001758 0.57056401
 0.56845375 0.56784734 0.56796282 0.56665763 0.56489808 0.5641638
 0.56552347 0.56509327 0.56383589 0.56376124 0.5633903  0.56308628
 0.56063777 0.55955363 0.56248142 0.56383127 0.56236906 0.56136258
 0.56186177 0.56136075 0.55809526 0.55600615 0.55441698 0.5509509
 0.54910707 0.54850129 0.54607014 0.54253061 0.54044988 0.53928042
 0.53640435 0.53246945 0.52952884 0.52969432 0.5286219  0.52651658
 0.5243656  0.52273972 0.52039799 0.51656112 0.51446042 0.51489386
 0.51527694 0.51576236 0.51690773 0.51818606 0.52007872 0.52418759
 0.52998904 0.53403135 0.53673744 0.53991336 0.543284   0.54532502
 0.54700111 0.54941625 0.55308823 0.5557178  0.55662786 0.55671328
 0.55675159 0.55823003 0.56249353 0.56728165 0.56927703 0.56979271
 0.56893642 0.56883859 0.56774491 0.56523536 0.56624038 0.566786

In [6]:
Net1 = Network(mtaskNet[1], traindata, testdata, 0.1, 0.0002)
netsize1 = Net1.net_size()
w_pro = np.random.randn(netsize1)
w_pro

array([-1.04007584e-01,  3.96208042e-01,  1.11577282e-01, -5.36818518e-01,
       -4.99181016e-01, -1.95412267e-01, -2.31503964e+00, -1.08785608e+00,
       -3.22658772e-01,  5.19624231e-01,  6.76805576e-01, -1.15787090e-01,
       -2.62240694e-02,  7.11887281e-01, -1.17366185e+00, -6.24813449e-01,
        1.22599727e+00, -1.51592863e+00, -2.51892255e+00, -8.56886543e-02,
       -2.59174064e-01, -4.33271874e-01,  6.03748827e-01,  1.62766800e+00,
       -1.11272589e+00,  6.91589350e-01, -6.91844978e-01,  1.24676463e+00,
       -1.13788764e+00, -1.79397683e+00, -1.10596185e+00, -8.31791776e-01,
        2.32971854e-04,  1.10933067e+00, -3.07807632e-02, -1.60135436e-01,
       -1.41850295e+00,  3.35073488e-01,  2.04968507e-01,  3.45903425e-01,
        1.36495220e-01, -8.18210230e-01,  7.27599909e-01, -1.17577235e+00,
        7.42451399e-01,  6.54500226e-01])

In [44]:
class MCMC:
	def __init__(self, samples, traindata, testdata, minPerf, mtaskNet):
		self.samples = samples  
		self.mtaskNet = mtaskNet  # mtaskNet = np.array([baseNet, secondnet]) -> to access each nn, mtaskNet[dims]
		self.traindata = traindata  #
		self.testdata = testdata 
		self.minPerf = minPerf
		# ----------------

	def rmse(self, predictions, targets):
		return np.sqrt(((predictions - targets) ** 2).mean())

	def likelihood_func(self, neuralnet, data, w, tausq, dims):
		topology = self.mtaskNet[dims]
		y = data[:, topology[0]]
		fx = neuralnet.evaluate_proposal(w)
		rmse = self.rmse(fx, y)
		n = y.shape[0]

		loss =( -(n/2) * np.log(2 * math.pi * tausq)) -( (1/(2*tausq)) * np.sum(np.square(y - fx))) #tausq is variance of error terms
		return [loss, fx, rmse]

	def prior_likelihood(self, sigma_squared, nu_1, nu_2, w, tausq, model_prior, dims):
		topology = self.mtaskNet[dims]
		h = topology[1]  # number hidden neurons
		d = topology[0]  # number input neurons
		part1 = -1 * ((d * h + h + 2) / 2) * np.log(sigma_squared)
		part2 = -1 / (2 * sigma_squared) * (sum(np.square(w)))
		part3 = -1 * np.log(model_prior)
		log_loss = part1 + part2 + part3 - (1 + nu_1) * np.log(tausq) - (nu_2 / tausq)
		return log_loss

	def v_jump(self, tausq, v, size_diff): #choose tausq well for better proposal
		log_v =( -(size_diff/2) * np.log(2 * math.pi * tausq)) -( (1/(2*tausq)) * np.sum(np.square(v))) #tausq is variance of error terms
		return log_v

	def sampler(self, dims1, dims2):
		#----- Model 1
		netw_1 = self.mtaskNet[dims1]		
		Net1 = Network(netw_1, self.traindata, self.testdata, 0.1, 0.0002)
		y_test_1 = self.testdata[:, netw_1[0]]
		y_train_1 = self.traindata[:, netw_1[0]]

		#----- Model 2
		netw_2 = self.mtaskNet[dims2]
		Net2 = Network(netw_2, self.traindata, self.testdata, 0.1, 0.0002)
		y_test_2 = self.testdata[:, netw_2[0]]
		y_train_2 = self.traindata[:, netw_2[0]]
		#----- Initialize MCMC
		samples = self.samples
		testsize = self.testdata.shape[0]
		trainsize = self.traindata.shape[0]
		fxtrain_samples = []
		fxtest_samples = []
		rmse_train = []
		rmse_test = []
		pos_w = []
		#--- HYPERPARAMETERS
		sigma_squared = 25
		nu_1 = 0
		nu_2 = 0
		model_prior = 1/2 #for both models 1 and 2

		
		naccept = 0
		for i in range(samples -1):
			w_size1 = (netw_1[0] * netw_1[1]) + (netw_1[1] * netw_1[2]) + netw_1[1] + netw_1[2]
			netw_2[1] = netw_1[1] + random.choice([-1, 1])
			w_size2 = (netw_2[0] * netw_2[1]) + (netw_2[1] * netw_2[2]) + netw_2[1] + netw_2[2]

			if w_size2 > w_size1:
				#--- Initial w
				w = np.random.rand(w_size1)
				#--- Propose error term
				pred_train = Net1.evaluate_proposal(w)
				eta = np.log(np.var(pred_train - y_train_1))
				tau_pro = np.exp(eta)
				#--- Calc current prior and likelihood
				[likelihood_current, pred_train, rmse_train_current] = self.likelihood_func(Net1, self.traindata, w, tau_pro, dims1) #w here used for bnn
				[pred_test_current, rmse_test_current] = Net1.test_proposal(w)	
				prior_current = self.prior_likelihood(sigma_squared, nu_1, nu_2, w, tau_pro, model_prior, dims1)
				#--- Propose jump vector v
				v = np.random.normal(0, 1, w_size2-w_size1)
				#--- Propose w
				w_proposal = np.lib.pad(w, (0, w_size2-w_size1), 'constant', constant_values=(0,v))
				#--- Calc proposed prior and likelihood
				[likelihood_proposal, pred_train_proposal, rmse_train_proposal] = self.likelihood_func(Net2, self.traindata, w_proposal, tau_pro, dims2)
				[pred_test_proposal, rmse_test_proposal] = Net2.test_proposal(w_proposal)				
				prior_proposal = self.prior_likelihood(sigma_squared, nu_1, nu_2, w_proposal, tau_pro, model_prior, dims2)
				log_v = self.v_jump(tau_pro, v, w_size2-w_size1)
				#--- Calc log posterior 
				log_posterior = (prior_proposal + likelihood_proposal) - (prior_current + likelihood_current + log_v) 
				try:
					mh_prob = min(1, math.exp(log_posterior))

				except OverflowError as e:
					mh_prob = 1

				u = random.uniform(0, 1)

				if u < mh_prob:
					# ACCEPT
					naccept += 1
					likelihood_current = likelihood_proposal
					prior_current = prior_proposal            
					w = w_proposal #assign proposed w to w which now has higher dim
					pos_w.append(w)

					fxtrain_samples.append(pred_train_proposal)
					fxtest_samples.append(pred_test_proposal)
					rmse_train.append(rmse_train_proposal)
					rmse_test.append(rmse_test_proposal)
				else:
					fxtrain_samples.append(pred_train)
					fxtest_samples.append(pred_test_current)
					rmse_train.append(rmse_train_current)
					rmse_test.append(rmse_test_current)

			else: #w_size2 < w_size1
				#--- Initial w
				w = np.random.rand(w_size1)				
				#--- Propose error term
				pred_train = Net1.evaluate_proposal(w)
				eta = np.log(np.var(pred_train - y_train_1))
				tau_pro = np.exp(eta)
				#--- Calc current prior and likelihood

				[likelihood_current, pred_train, rmse_train_current] = self.likelihood_func(Net1, traindata, w, tau_pro, dims1) #w here used for bnn
				[pred_test_current, rmse_test_current] = Net1.test_proposal(w)	
				prior_current = self.prior_likelihood(sigma_squared, nu_1, nu_2, w, tau_pro, model_prior, dims1)
				#--- Propose jump vector v
				v = np.random.normal(0, 1, w_size1-w_size2)
				#--- Propose w
				w_proposal_down = w[0:w_size2] #propose w_down that has lower dim
				#--- Calc proposed prior and likelihood
				[likelihood_proposal, pred_train_proposal, rmse_train_proposal] = self.likelihood_func(Net2, self.traindata, w_proposal_down, tau_pro, dims2)
				[pred_test_proposal, rmse_test_proposal] = Net2.test_proposal(w_proposal_down)	
				prior_proposal = self.prior_likelihood(sigma_squared, nu_1, nu_2, w_proposal_down, tau_pro, model_prior, dims2)
				log_v = self.v_jump(tau_pro, v, w_size1-w_size2)
				#--- Calc log posterior 

				log_posterior_down = (prior_proposal + likelihood_proposal + log_v) - (prior_current + likelihood_current) 
				try:
					mh_prob_down = min(1, math.exp(log_posterior_down))
				except OverflowError as e:
					mh_prob_down = 1
				u_down = random.uniform(0, 1)
				if u_down < mh_prob_down:
					naccept += 1
					likelihood_current = likelihood_proposal
					prior_current = prior_proposal
					w = w_proposal_down
					pos_w.append(w)					
					
					fxtrain_samples.append(pred_train_proposal)
					fxtest_samples.append(pred_test_proposal)
					rmse_train.append(rmse_train_proposal)
					rmse_test.append(rmse_test_proposal)
				else:
					fxtrain_samples.append(pred_train)
					fxtest_samples.append(pred_test_current)
					rmse_train.append(rmse_test_current)
					rmse_test.append(rmse_test_current)

			accept_ratio = naccept / (samples)
			print(i, 'jump from model 1 with size', w_size1,'to model 2 with size', w_size2, np.shape(pos_w), '{:.1%}'.format(accept_ratio), 'is accepted')
		return (pos_w, fxtrain_samples, fxtest_samples, rmse_train, rmse_test, accept_ratio)


In [45]:
mcmc = MCMC(5, traindata, testdata, 0.002, mtaskNet)
mcmc.sampler(0,1)

0 jump from model 1 with size 46 to model 2 with size 55 (1, 55) 20.0% is accepted
1 jump from model 1 with size 46 to model 2 with size 55 (2, 55) 40.0% is accepted
2 jump from model 1 with size 46 to model 2 with size 55 (3, 55) 60.0% is accepted
3 jump from model 1 with size 46 to model 2 with size 37 (4,) 80.0% is accepted


([array([ 0.93413042,  0.02153626,  0.42628011,  0.83185607,  0.5921436 ,
          0.86116579,  0.95929247,  0.55330767,  0.24214761,  0.61754579,
          0.97797625,  0.45235969,  0.4397006 ,  0.94045525,  0.42528524,
          0.71239511,  0.89792736,  0.79185078,  0.58907723,  0.24656741,
          0.45141387,  0.04191473,  0.16060217,  0.66697878,  0.78581262,
          0.5718963 ,  0.6821201 ,  0.72497889,  0.88325066,  0.44543451,
          0.22228925,  0.83393632,  0.80903435,  0.56004288,  0.7497558 ,
          0.25866075,  0.65535267,  0.63417146,  0.66719999,  0.99852049,
          0.88334274,  0.13343402,  0.59209643,  0.04599493,  0.41223691,
          0.00831191, -1.04845809, -0.51526209,  0.6009934 , -0.43377682,
         -0.7592915 , -0.53201937,  0.40220082,  0.23158068,  1.55397946]),
  array([ 0.93800362,  0.57345578,  0.57836027,  0.40252798,  0.088081  ,
          0.36234575,  0.44142622,  0.35214377,  0.19418664,  0.3898911 ,
          0.78743874,  0.10040763,  

In [52]:
[pos_w, fxtrain_samples, fxtest_samples, rmse_train, rmse_test, accept_ratio] = mcmc.sampler(0,1)
fx_mu = np.mean(fxtest_samples, axis=0)
#np.shape(fxtrain_samples)
fx_mu

0 jump from model 1 with size 46 to model 2 with size 55 (1, 55) 20.0% is accepted
1 jump from model 1 with size 46 to model 2 with size 37 (2,) 40.0% is accepted
2 jump from model 1 with size 46 to model 2 with size 37 (3,) 60.0% is accepted
3 jump from model 1 with size 46 to model 2 with size 37 (4,) 80.0% is accepted


/anaconda3/envs/env_pytorch/lib/python3.6/site-packages/numpy/lib/arraypad.py:490: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray
  x = np.array(x)
/anaconda3/envs/env_pytorch/lib/python3.6/site-packages/numpy/core/_asarray.py:83: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray
  return array(a, dtype, copy=False, order=order)


array([0.76564911, 0.76783534, 0.76848669, 0.76615914, 0.76246298,
       0.75723088, 0.75244713, 0.75058882, 0.74969417, 0.74970431,
       0.74734306, 0.74395493, 0.73925917, 0.73243837, 0.72518238,
       0.71637291, 0.70961523, 0.70245458, 0.69565375, 0.68967547,
       0.68503636, 0.68354855, 0.6833897 , 0.68350778, 0.68336521,
       0.68142007, 0.67887323, 0.67630638, 0.67395809, 0.67285457,
       0.67157103, 0.67137306, 0.67203427, 0.6723874 , 0.67288924,
       0.67223958, 0.67170127, 0.67107231, 0.67083502, 0.67143599,
       0.67277049, 0.67554325, 0.68005927, 0.6876672 , 0.69683155,
       0.7072705 , 0.71740858, 0.72784295, 0.73785785, 0.74687701,
       0.75422583, 0.75978818, 0.76441959, 0.76740503, 0.76991735,
       0.771641  , 0.77372123, 0.77558776, 0.77744449, 0.77843547,
       0.77919853, 0.77955093, 0.7787842 , 0.7777446 , 0.77517715,
       0.7731256 , 0.77061106, 0.7674875 , 0.76458555, 0.76128696,
       0.7589223 , 0.75702942, 0.75356821, 0.74941499, 0.74458